In [1]:
%pip install -q kagglehub transformers torch bitsandbytes requests tqdm

In [2]:
import os
import json
import random
import pandas as pd
import kagglehub

# Output file targets
FINAL_FILE = "hinglish_textosql_4500.json"
FINAL_CSV_FILE = "hinglish_textosql_4500.csv"
BACKUP_FILE = "hinglish_textosql_backup.json"

print("Downloading dataset from Kaggle...")
path = kagglehub.dataset_download("mohammadnouralawad/spider-text-sql")
csv_path = os.path.join(path, "spider_text_sql.csv")

df = pd.read_csv(csv_path)
sampled_df = df.sample(n=1500, random_state=42).reset_index(drop=True)

completed_dataset = []
print(f"Fresh session initialized. Ready to process {len(sampled_df)} rows.")

Using Colab cache for faster access to the 'spider-text-sql' dataset.
Fresh session initialized. Ready to process 1500 rows.


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-7B-Instruct"

print("Loading Qwen 2.5 Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading Qwen 2.5 Model weights (this may take a couple of minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto" # Automatically loads layers onto your active GPU
)
print("Model loaded and live on GPU!")

Loading Qwen 2.5 Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading Qwen 2.5 Model weights (this may take a couple of minutes)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded and live on GPU!


In [4]:
import os
import json
import random
import pandas as pd
import kagglehub

# Dataset destination files
FINAL_FILE = "hinglish_textosql_4500.json"
FINAL_CSV_FILE = "hinglish_textosql_4500.csv"
BACKUP_FILE = "hinglish_textosql_backup.json"

print("Downloading dataset from Kaggle...")
path = kagglehub.dataset_download("mohammadnouralawad/spider-text-sql")
csv_path = os.path.join(path, "spider_text_sql.csv")

# Load and sample strictly with seed=42
df = pd.read_csv(csv_path)
sampled_df = df.sample(n=1500, random_state=42).reset_index(drop=True)

# Instantiating a clean empty list for a fresh pipeline start
completed_dataset = []
print(f" Fresh run initiated. Ready to process {len(sampled_df)} items.")

Using Colab cache for faster access to the 'spider-text-sql' dataset.
 Fresh run initiated. Ready to process 1500 items.


In [5]:
def generate_hinglish_variants(english_text):
    system_instruction = "You are a linguist translating English text into Hinglish (Hindi-English code-mixed style)."

    prompt = f"""Translate the following English sentence into three distinct Hinglish variants.

English: "{english_text}"

You must output exactly one single line containing the three translations separated by " ||| " in this exact order:
LIGHT_VARIANT ||| NATURAL_VARIANT ||| HEAVY_VARIANT

Rules:
1. LIGHT_VARIANT: Around 70% English, 30% Hindi words.
2. NATURAL_VARIANT: Balanced 50-50 code-mixed style.
3. HEAVY_VARIANT: Around 30% English, 70% Hindi words written in Roman script (Latin script).
4. Provide NO introduction, NO explanations, and NO markdown ticks. Just the raw single line with delimiters.
"""

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    try:
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=120,
                temperature=0.2,
                do_sample=False,
                eos_token_id=[151645, 151643]
            )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return response.strip()
    except Exception as e:
        print(f"\nInference Warning: {e}")
        return None

In [6]:
from tqdm import tqdm

print("Generating variants...")

for idx, row in tqdm(sampled_df.iterrows(), total=len(sampled_df)):
    english_query = row['text_query']
    sql_command = row['sql_command']

    success = False
    retries = 2

    while retries > 0 and not success:
        model_output = generate_hinglish_variants(english_query)

        if model_output:
            parts = [p.strip() for p in model_output.split("|||")]

            if len(parts) == 3:
                completed_dataset.append({
                    "english": english_query,
                    "hinglish_light": parts[0].replace('"', ''),
                    "hinglish_natural": parts[1].replace('"', ''),
                    "hinglish_hindi_heavy": parts[2].replace('"', ''),
                    "sql": sql_command
                })
                success = True
            else:
                retries -= 1
        else:
            retries -= 1

    # Auto-save dynamic checkpoints every 50 records processed
    if len(completed_dataset) % 50 == 0:
        with open(BACKUP_FILE, "w", encoding="utf-8") as f:
            json.dump(completed_dataset, f, indent=2, ensure_ascii=False)

Generating variants...


100%|██████████| 1500/1500 [2:15:30<00:00,  5.42s/it]


In [8]:
import os

print("Files in current directory:")
print(os.listdir('.'))

if os.path.exists("hinglish_textosql_backup.json"):
    print("\nFound the backup file! Let's rename and secure it.")
    os.rename("hinglish_textosql_backup.json", "hinglish_textosql_4500.json")

if 'completed_dataset' in locals() and len(completed_dataset) > 0:
    import json
    import pandas as pd
    print(f"\nIn-memory dataset found with {len(completed_dataset)} rows. Writing clean files now...")
    with open("hinglish_textosql_4500.json", "w", encoding="utf-8") as f:
        json.dump(completed_dataset, f, indent=2, ensure_ascii=False)
    pd.DataFrame(completed_dataset).to_csv("hinglish_textosql_4500.csv", index=False, encoding="utf-8")
    print("Files forced to disk successfully!")

Files in current directory:
['.config', 'hinglish_textosql_backup.json', 'sample_data']

Found the backup file! Let's rename and secure it.

In-memory dataset found with 987 rows. Writing clean files now...
Files forced to disk successfully!


In [9]:
from google.colab import files

files.download("hinglish_textosql_4500.json")
files.download("hinglish_textosql_4500.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import pandas as pd

df = pd.read_csv("hinglish_textosql_4500.csv")

base_queries = len(df)
total_variants = base_queries * 3

print(f"Base English/SQL pairs: {base_queries}")
print(f"Total Hinglish query variants generated: {total_variants}")

Base English/SQL pairs: 987
Total Hinglish query variants generated: 2961


In [11]:
import json

try:
    with open("hinglish_textosql_4500.json", "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"Base English/SQL records in JSON: {len(data)}")
    print(f"Total Hinglish query variants inside: {len(data) * 3}")

except FileNotFoundError:
    print("The file 'hinglish_textosql_4500.json' was not found in this directory.")
except json.JSONDecodeError:
    print("The file exists but contains invalid JSON formatting.")

Base English/SQL records in JSON: 987
Total Hinglish query variants inside: 2961


In [13]:
print(f"Current rows in memory list: {len(completed_dataset)}")
if len(completed_dataset) > 0:
    print("Last row in memory look:", completed_dataset[-1])

Current rows in memory list: 994
Last row in memory look: {'english': 'what are the order id and customer id of the oldest order?', 'hinglish_light': 'what are the order id aur oldest order ka customer id?', 'hinglish_natural': 'oldest order ka order id aur customer id kya hain?', 'hinglish_hindi_heavy': 'kya aapka oldest order ka order id aur customer id pata chala hai?', 'sql': 'SELECT order_id ,  customer_id FROM orders ORDER BY date_order_placed LIMIT 1'}


In [15]:
from tqdm import tqdm
import os
import json
import warnings

warnings.filterwarnings("ignore", message="The following `model_kwargs` are not used by the model")
warnings.filterwarnings("ignore", message="The following generation flags are not valid")

current_count = len(completed_dataset)
print(f"Resuming cleanly from row {current_count}. Remaining rows to process: {1500 - current_count}")

remaining_df = sampled_df.iloc[current_count:]

for idx, row in tqdm(remaining_df.iterrows(), total=len(remaining_df)):
    english_query = row['text_query']
    sql_command = str(row['sql_command']).strip()

    success = False
    retries = 2

    while retries > 0 and not success:
        model_output = generate_hinglish_variants(english_query)

        if model_output:
            parts = [p.strip() for p in model_output.split("|||")]
            if len(parts) == 3:
                completed_dataset.append({
                    "english": english_query,
                    "hinglish_light": parts[0].replace('"', ''),
                    "hinglish_natural": parts[1].replace('"', ''),
                    "hinglish_hindi_heavy": parts[2].replace('"', ''),
                    "sql": sql_command
                })
                success = True
            else:
                retries -= 1
        else:
            retries -= 1

    if len(completed_dataset) % 30 == 0:
        with open("hinglish_textosql_4500.json", "w", encoding="utf-8") as f:
            json.dump(completed_dataset, f, indent=2, ensure_ascii=False)

with open("hinglish_textosql_4500.json", "w", encoding="utf-8") as f:
    json.dump(completed_dataset, f, indent=2, ensure_ascii=False)

import pandas as pd
pd.DataFrame(completed_dataset).to_csv("hinglish_textosql_4500.csv", index=False, encoding="utf-8")
print(f"\n Mission Accomplished! Final dataset contains {len(completed_dataset)} rows.")

Resuming cleanly from row 1309. Remaining rows to process: 191


100%|██████████| 191/191 [17:51<00:00,  5.61s/it]


 Mission Accomplished! Final dataset contains 1430 rows.


In [16]:
import json
import pandas as pd

print("VERIFYING COMPLETED DATASET COUNTS\n")

try:
    df = pd.read_csv("hinglish_textosql_4500.csv")
    csv_base = len(df)
    csv_total_variants = csv_base * 3
    print(f"CSV File Verification:")
    print(f"   - Base English/SQL pairs: {csv_base}")
    print(f"   - Total Hinglish variations: {csv_total_variants}")
except Exception as e:
    print(f"Error reading CSV: {e}")

try:
    with open("hinglish_textosql_4500.json", "r", encoding="utf-8") as f:
        json_data = json.load(f)
    json_base = len(json_data)
    json_total_variants = json_base * 3
    print(f"JSON File Verification:")
    print(f"   - Base English/SQL records: {json_base}")
    print(f"   - Total Hinglish variations: {json_total_variants}")
except Exception as e:
    print(f"Error reading JSON: {e}")

# Final confirmation check
if csv_base == 1500 and json_base == 1500:
    print("\nBoth files hit the exact target of 1,500 base rows (4,500 total variants).")
else:
    print(f"\nCount discrepancy or incomplete run. Current base count is {csv_base}/1500.")

VERIFYING COMPLETED DATASET COUNTS

CSV File Verification:
   - Base English/SQL pairs: 1430
   - Total Hinglish variations: 4290
JSON File Verification:
   - Base English/SQL records: 1430
   - Total Hinglish variations: 4290

Count discrepancy or incomplete run. Current base count is 1430/1500.


In [17]:
from google.colab import files

files.download("hinglish_textosql_4500.json")
files.download("hinglish_textosql_4500.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
from tqdm import tqdm
import os
import json
import warnings

warnings.filterwarnings("ignore", message="The following `model_kwargs` are not used by the model")
warnings.filterwarnings("ignore", message="The following generation flags are not valid")

current_count = len(completed_dataset)
print(f"Fetching the final stretch! Resuming from row {current_count}. Remaining rows: {1500 - current_count}")

final_remaining_df = sampled_df.iloc[current_count:]

for idx, row in tqdm(final_remaining_df.iterrows(), total=len(final_remaining_df)):
    english_query = row['text_query']
    sql_command = str(row['sql_command']).strip()

    success = False
    retries = 2

    while retries > 0 and not success:
        model_output = generate_hinglish_variants(english_query)

        if model_output:
            parts = [p.strip() for p in model_output.split("|||")]
            if len(parts) == 3:
                completed_dataset.append({
                    "english": english_query,
                    "hinglish_light": parts[0].replace('"', ''),
                    "hinglish_natural": parts[1].replace('"', ''),
                    "hinglish_hindi_heavy": parts[2].replace('"', ''),
                    "sql": sql_command
                })
                success = True
            else:
                retries -= 1
        else:
            retries -= 1

with open("hinglish_textosql_4500.json", "w", encoding="utf-8") as f:
    json.dump(completed_dataset, f, indent=2, ensure_ascii=False)

import pandas as pd
pd.DataFrame(completed_dataset).to_csv("hinglish_textosql_4500.csv", index=False, encoding="utf-8")
print(f"\nFinal dataset contains exactly {len(completed_dataset)} rows.")

Fetching the final stretch! Resuming from row 1430. Remaining rows: 70


100%|██████████| 70/70 [07:16<00:00,  6.23s/it]


Final dataset contains exactly 1469 rows.


In [19]:
from google.colab import files

files.download("hinglish_textosql_4500.json")
files.download("hinglish_textosql_4500.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>